# Validate GFM `reference_water_mask` code mapping against EODC COGs

**Goal.** Treat the fetched EODC STAC COGs as the **source of truth** for the GFM
`reference_water_mask` pixel codes, and confirm the mapping Atlantis assumes:

| Code | Meaning (GFM PDD Table 20) |
| --- | --- |
| `0`   | no water |
| `1`   | permanent water |
| `2`   | seasonal water (per month) |
| `255` | no data |

This notebook operationalises the **month-stability test** referenced in
`src/atlantis/fetchers/gfm/layers.py`: code `1` is byte-identical across the
monthly reference-water masks (permanent), while code `2` varies by month
(seasonal). An earlier assumption that the permanent/seasonal codes were
swapped was a bug, now corrected in `atlantis.fetchers.gfm.layers`.

It mirrors `validate_viirs_legend.ipynb` and `validate_modis_legend.ipynb`.

> Internet access is required — every COG read is via `/vsicurl/`. No bytes are
> written to disk; we stream small windows only.


## 1. Setup

In [ ]:
from __future__ import annotations

import os
import sys
from collections import defaultdict
from datetime import date
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
# Walk up until we find the repo marker (pyproject.toml).
candidate = NOTEBOOK_DIR
while candidate != candidate.parent and not (candidate / 'pyproject.toml').exists():
    candidate = candidate.parent
REPO_ROOT = candidate
SRC = REPO_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f'REPO_ROOT = {REPO_ROOT}')

# Make GDAL behave well with /vsicurl/ over slow links.
os.environ.setdefault('GDAL_HTTP_TIMEOUT', '30')
os.environ.setdefault('GDAL_HTTP_MAX_RETRY', '3')
os.environ.setdefault('CPL_VSIL_CURL_CHUNK_SIZE', '524288')  # 512 KiB


In [ ]:
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from shapely.geometry import box

from atlantis.fetchers.gfm.backend import DEFAULT_GFM_STAC_URL, GFM_COLLECTION_ID
from atlantis.fetchers.gfm.layers import (
    GFM_DRY,
    GFM_LAND,
    GFM_NODATA,
    GFM_PERMANENT_WATER,
    GFM_SEASONAL_WATER,
)

print('EODC STAC endpoint :', DEFAULT_GFM_STAC_URL)
print('GFM collection     :', GFM_COLLECTION_ID)
print('Atlantis constants :')
for name in ('GFM_LAND', 'GFM_PERMANENT_WATER', 'GFM_SEASONAL_WATER', 'GFM_NODATA'):
    print(f'  {name:>24} = {globals()[name]}')


## 2. Reference legends, side-by-side

Before touching the network we capture the reference legends as Python tables so
we can diff them against the **authoritative** codes the live COGs report. None
of these is the source of truth — the fetched COGs are.

- **PDD legend** — GFM PDD Table 20 (the public product definition).
- **Atlantis decoder** — pulled directly from the live constants in
  `atlantis.fetchers.gfm.layers`.


In [ ]:
# --- Reference 1: GFM PDD Table 20 ---
PDD_LEGEND: dict[int, str] = {
    0:   'no water',
    1:   'permanent water',
    2:   'seasonal water (per month)',
    255: 'no data',
}

# --- Reference 2: live Atlantis decoder constants ---
ATLANTIS_DECODER: dict[int, str] = {
    GFM_LAND:            'no water',
    GFM_PERMANENT_WATER: 'permanent water',
    GFM_SEASONAL_WATER:  'seasonal water',
    GFM_NODATA:          'nodata',
}

# Sanity check: the two legends must agree on every code.
mismatch = {c: (PDD_LEGEND.get(c), ATLANTIS_DECODER.get(c))
            for c in set(PDD_LEGEND) | set(ATLANTIS_DECODER)
            if PDD_LEGEND.get(c) != ATLANTIS_DECODER.get(c)}
assert not mismatch, f'PDD vs Atlantis legend mismatch: {mismatch}'
print('PDD legend and Atlantis decoder agree on all codes.')
pd.DataFrame(sorted(PDD_LEGEND.items()), columns=['code', 'meaning'])


## 3. Search EODC STAC for GFM items across several months

We use the **Valencia 2024** flood event bbox (covers the Albufera lake — a
permanent-water body — plus surrounding seasonal wetlands), and request items
across a multi-month window so the month-stability test has several monthly
reference-water masks to compare.

The reference-water mask in GFM is a *monthly* climatology, so different
acquisition dates within the same calendar month share the same mask; different
months differ in the seasonal class (`2`) only.


In [ ]:
from pystac_client import Client

# Valencia 2024 bbox (from docs/gfm/overview.md). Covers Albufera + wetlands.
BBOX = (-1.5, 38.8, 0.5, 40.0)
# Multi-month window so we get several distinct monthly masks.
START = date(2024, 10, 1)
END = date(2025, 3, 31)

aoi = box(*BBOX)
catalog = Client.open(DEFAULT_GFM_STAC_URL)
search = catalog.search(
    max_items=1000,
    collections=GFM_COLLECTION_ID,
    intersects=aoi,
    datetime=(START.isoformat(), END.isoformat()),
)
items = list(search.get_items())
print(f'Found {len(items)} GFM items over Valencia, {START} -> {END}')

# Group by calendar month (YYYY-MM). The reference_water_mask is monthly, so we
# only need one representative item per month.
by_month: dict[str, object] = {}
for it in items:
    dt = it.datetime
    if dt is None:
        continue
    key = dt.strftime('%Y-%m')
    by_month.setdefault(key, it)
months = sorted(by_month)
print(f'Distinct months with coverage: {months}')
assert len(months) >= 2, 'Need at least 2 months for the stability test.'


## 4. Read `reference_water_mask` for one item per month

For each month we open the `reference_water_mask` asset via `/vsicurl/` and read
a fixed window. We pick the window from the first item's footprint and reuse the
same pixel offsets for every month so the arrays are directly comparable.


In [ ]:
ASSET_KEY = 'reference_water_mask'

def asset_href(item) -> str:
    """Return the /vsicurl/ href for the reference_water_mask asset."""
    asset = item.assets.get(ASSET_KEY)
    if asset is None:
        # Some STAC catalogs expose assets under alternate keys; fall back to a
        # case-insensitive match on the asset title/id.
        for k, a in item.assets.items():
            if k.lower() == ASSET_KEY or (a.title or '').lower() == ASSET_KEY:
                asset = a
                break
    assert asset is not None, (
        f'Item {item.id} has no {ASSET_KEY!r} asset. Available: {list(item.assets)}'
    )
    return asset.href

# Inspect the first item's asset keys + the band's codes/tags.
first = by_month[months[0]]
print('First item:', first.id, '| datetime:', first.datetime)
print('Asset keys:', list(first.assets))
href0 = asset_href(first)
print('reference_water_mask href:', href0)
with rasterio.open(f'/vsicurl/{href0}') as src:
    print('shape:', src.shape, '| dtype:', src.dtypes, '| nodata:', src.nodata)
    # Tags often carry the code legend (analogous to VIIRS WaterDetection tag).
    print('band tags:', dict(src.tags(1)))


In [ ]:
# Choose a centred 512x512 window from the first item and reuse it for every month.
with rasterio.open(f'/vsicurl/{href0}') as src:
    h, w = src.shape
    win = Window(col_off=max(0, w // 2 - 256), row_off=max(0, h // 2 - 256), width=512, height=512)
    win = win.round_offsets(op='floor').round_lengths(op='floor')
    transform_win = src.window_transform(win)
print('window:', win, '| covers (lon,lat) near:', transform_win.c, transform_win.f)

def clipped_window(win: Window, shape: tuple[int, int]) -> Window:
    """Clip a window to an item's (height, width) so reads never overrun."""
    h, w = shape
    col_off = max(0, int(win.col_off))
    row_off = max(0, int(win.row_off))
    width = min(int(win.width), w - col_off)
    height = min(int(win.height), h - row_off)
    return Window(col_off=col_off, row_off=row_off, width=width, height=height)

masks: dict[str, np.ndarray] = {}
for m in months:
    item = by_month[m]
    href = asset_href(item)
    with rasterio.open(f'/vsicurl/{href}') as src:
        local_win = clipped_window(win, src.shape)
        arr = src.read(1, window=local_win)
    masks[m] = arr
    uniq = np.unique(arr, return_counts=True)
    print(f'{m}: unique codes = {dict(zip(uniq[0].tolist(), uniq[1].tolist()))}')


## 5. Month-stability test

The defining property of the GFM reference-water mask (per `layers.py:35-41`):

- **Code `1` (permanent)** must be byte-identical across months — a permanent
  water body does not move month to month.
- **Code `2` (seasonal)** is *expected* to vary across months — that is what
  makes it seasonal rather than permanent.

If the codes were swapped (the old bug), code `2` would be the stable one and
code `1` would vary — the test below would fail loudly.


In [ ]:
def stability_for_code(code: int) -> tuple[bool, int]:
    """Return (is_stable_across_months, n_months_containing_code)."""
    binary = {m: (arr == code) for m, arr in masks.items()}
    present = {m: b for m, b in binary.items() if b.any()}
    if len(present) < 2:
        return True, len(present)  # not enough months to detect variation
    ref_m, ref_b = next(iter(present.items()))
    is_stable = all(np.array_equal(ref_b, b) for b in present.values())
    return is_stable, len(present)

rows = []
for code, label in ((GFM_PERMANENT_WATER, 'permanent (1)'),
                    (GFM_SEASONAL_WATER, 'seasonal (2)')):
    stable, n = stability_for_code(code)
    rows.append({'code': code, 'class': label, 'months_present': n, 'stable_across_months': stable})
stability = pd.DataFrame(rows)
stability


In [ ]:
# Assertions encoding the corrected mapping.
# - permanent (1) must be stable (identical across months)
# - seasonal  (2) must vary across months (when present in >=2 months)
perm = stability.loc[stability['code'] == GFM_PERMANENT_WATER].iloc[0]
seas = stability.loc[stability['code'] == GFM_SEASONAL_WATER].iloc[0]

assert bool(perm['stable_across_months']), (
    'Code 1 (permanent) is NOT stable across months — mapping may be swapped!'
)
print('PASS: code 1 (permanent) is byte-identical across months.')

if int(seas['months_present']) >= 2:
    assert not bool(seas['stable_across_months']), (
        'Code 2 (seasonal) is stable across months — expected variation. '
        'Either no seasonal water in this window, or the mapping is swapped.'
    )
    print('PASS: code 2 (seasonal) varies across months, as expected.')
else:
    print(f'SKIP: seasonal code 2 present in only {int(seas["months_present"])} month(s); '
          'choose a window with seasonal wetlands to exercise this branch.')


## 6. Cross-check observed codes vs PDD and Atlantis

Every code that appears in the fetched COGs must be declared in both the PDD
legend and the Atlantis decoder constants. Any unknown code is a regression.


In [ ]:
observed = set()
for arr in masks.values():
    observed.update(np.unique(arr).tolist())
observed.discard(GFM_NODATA)  # nodata may or may not appear in the window
known = set(PDD_LEGEND)
unknown = observed - known
print('Observed codes:', sorted(observed))
print('Known codes   :', sorted(known))
print('Unknown codes :', sorted(unknown))
assert not unknown, f'Observed codes not in PDD/Atlantis legend: {sorted(unknown)}'
print('PASS: all observed codes are declared in the PDD + Atlantis legends.')


## 7. Conclusion

If all assertions above pass, the fetched EODC COGs confirm the corrected GFM
`reference_water_mask` mapping:

- `1 = permanent water` (stable across monthly masks),
- `2 = seasonal water` (varies by month),

in agreement with GFM PDD Table 20 and with `atlantis.fetchers.gfm.layers`.
The earlier '2 = permanent' assumption was a bug, now corrected in code and in
`docs/gfm/**` and `docs/layers.md`.

Re-run this notebook whenever the GFM STAC collection or the registry constants
change; treat the fetched COGs as the ground truth.
